#<font color="Green">**Notebook Purpose**</font>

This notebook constructs a comparison table of demographic and baseline clinical characteristics between the **Primary Analytic Cohort** and the **Early Disengagement Clusters**. This addresses Reviewer 1 Comment 3, Reviewer 2 Comment 3, and Reviewer 5's concerns about potential selection bias introduced by the post-hoc separation of disengagement clusters.

**What this notebook produces:**
- A side-by-side comparison table with columns for Primary Cohort, Disengagement Cohort, and Standardized Mean Differences (SMDs)
- Baseline HbA1c and BMI are defined as the closest recorded value **before** each patient's first GLM prescription within 2019 (consistent with the revised Table 1 methodology)
- SMDs are used instead of p-values because with large sample sizes (~9,000 per group), even trivially small differences reach statistical significance

**SMD interpretation:** < 0.10 = negligible, 0.10–0.20 = small, > 0.20 = meaningful

---

###<font color="Red"> Required Data </font>

To run the code blocks in this notebook, you will need the following **cleaned** CSVs (output from `CohortDatasetCreation.ipynb`):

1. **`medication_info.csv`** — columns: `patient_id`, `start_date`, `ingredient`, `medication_class`
2. **`lab_results.csv`** — columns: `patient_id`, `date`, `lab_result_num_val` (HbA1c, cleaned to 3.5–20% range)
3. **`BMI_vital_signs.csv`** — columns: `patient_id`, `date`, `value` (BMI)
4. **`patient_demographics.csv`** — columns: `patient_id`, `sex`, `race/ethnicity`, `year_of_birth`, `month_year_death`, `patient_regional_location`
5. **`patient_comorbidities.csv`** — columns: `patient_id`, `date`, `HF`, `CKD`
6. **`early_dropout_patients.pkl`** — dict of `{cluster_id: [patient_ids]}`

Additionally, after running **Section 2**, you must take the exported `patient_index_all.csv` to the `get_closest_continuous_value_after_t0` notebook to extract baseline values. You will then return with:

7. **`baseline_a1c_all.csv`** — output from `get_closest_continuous_value_after_t0`
8. **`baseline_bmi_all.csv`** — output from `get_closest_continuous_value_after_t0`

## Section 1 — Imports & Data Loading

In [ ]:
!pip install xlsxwriter

In [ ]:
import pandas as pd
import numpy as np
import pickle as pkl

from xlsxwriter import Workbook

In [ ]:
medication_info = pd.read_csv('/content/medication_info.csv')
lab_results = pd.read_csv('/content/lab_results.csv')
BMI_vital_signs = pd.read_csv('/content/BMI_vital_signs.csv')
patient_demographics = pd.read_csv('/content/patient_demographics.csv')
patient_comorbidities = pd.read_csv('/content/patient_comorbidities.csv')

with open('/content/early_dropout_patients.pkl', 'rb') as f:
    early_dropout_patients = pkl.load(f)

In [ ]:
# Parse dates
medication_info['start_date'] = pd.to_datetime(medication_info['start_date'], errors='coerce')
lab_results['date'] = pd.to_datetime(lab_results['date'], errors='coerce')
BMI_vital_signs['date'] = pd.to_datetime(BMI_vital_signs['date'], errors='coerce')

## Section 2 — Build Patient Index (First GLM Date) for ALL Patients

Unlike Table 1 construction, we do **not** exclude early dropout patients here. We need the first GLM date for every patient so that baseline extraction can be run for both groups.

In [ ]:
patient_index = (
    medication_info
    .sort_values('start_date')
    .groupby('patient_id', as_index=False)['start_date']
    .first()
    .rename(columns={'start_date': 't0'})
)

print(f"Patient index created: {len(patient_index):,} patients (both groups)")
print(f"First GLM date range: {patient_index['t0'].min()} to {patient_index['t0'].max()}")
patient_index.head()

### <font color="Red">EXPORT: `patient_index_all.csv`</font>

Export this file and take it to the `get_closest_continuous_value_after_t0` notebook.

**For baseline HbA1c extraction**, use these parameters:
- `index_df` = `patient_index_all.csv`
- `values_df` = `lab_results.csv`, **pre-filtered to 2019** (`date >= '2019-01-01'` and `date < '2020-01-01'`)
- `target_days_from_t0 = 0`
- `days_before_target_date = 183`
- `days_after_target_date = 0`
- `value_name = 'baseline_a1c'`
- `value_col = 'lab_result_num_val'`
- `value_date_col = 'date'`
- `prefer_value = 'min'`

**For baseline BMI extraction**, use the same parameters except:
- `values_df` = `BMI_vital_signs.csv`, **pre-filtered to 2019**
- `value_name = 'baseline_bmi'`
- `value_col = 'value'`

Export the results as `baseline_a1c_all.csv` and `baseline_bmi_all.csv`, then return to **Section 3** below.

In [ ]:
patient_index.to_csv('/content/patient_index_all.csv', index=False)
print("Exported patient_index_all.csv")

---

## <font color="Orange">BREAK POINT</font>

**Go to `get_closest_continuous_value_after_t0` notebook now.**

1. Load `patient_index_all.csv` as your `index_df`
2. Pre-filter `lab_results` to 2019 dates only before passing as `values_df`
3. Run extraction with the parameters above for both HbA1c and BMI
4. Export as `baseline_a1c_all.csv` and `baseline_bmi_all.csv`
5. Return here and continue with Section 3

---

## Section 3 — Load Baseline Extraction Results & Split into Groups

In [ ]:
baseline_a1c = pd.read_csv('/content/baseline_a1c_all.csv')
baseline_bmi = pd.read_csv('/content/baseline_bmi_all.csv')

print(f"Baseline HbA1c records loaded: {len(baseline_a1c):,}")
print(f"  - with a value: {baseline_a1c['baseline_a1c_value'].notna().sum():,}")
print(f"  - missing: {baseline_a1c['baseline_a1c_value'].isna().sum():,}")
print()
print(f"Baseline BMI records loaded: {len(baseline_bmi):,}")
print(f"  - with a value: {baseline_bmi['baseline_bmi_value'].notna().sum():,}")
print(f"  - missing: {baseline_bmi['baseline_bmi_value'].isna().sum():,}")

In [ ]:
# Build dropout ID set
dropout_ids = set()
for _, id_list in early_dropout_patients.items():
    dropout_ids.update([str(x) for x in id_list])

print(f"Early disengagement patient IDs: {len(dropout_ids):,}")

# Tag every dataframe with group membership
def tag_group(df, pid_col='patient_id'):
    df = df.copy()
    df[pid_col] = df[pid_col].astype(str)
    df['group'] = np.where(df[pid_col].isin(dropout_ids), 'Disengagement', 'Primary')
    return df

dem = tag_group(patient_demographics)
a1c = tag_group(baseline_a1c)
bmi = tag_group(baseline_bmi)
com = tag_group(patient_comorbidities)

n_primary = (dem['group'] == 'Primary').sum()
n_disengage = (dem['group'] == 'Disengagement').sum()
print(f"\nPrimary cohort: {n_primary:,}")
print(f"Disengagement cohort: {n_disengage:,}")
print(f"Total: {n_primary + n_disengage:,}")

## Section 4 — Configuration & Helper Functions

In [ ]:
# =========================
# Configuration (matches Table1_Construction)
# =========================
PCT_DECIMALS = 1

RACE_INPUT_LEVELS = ['White', 'Hispanic', 'Asian', 'Black', 'Other']
RACE_OUTPUT_MAP = {
    'Black':    'African American/Black, NH',
    'Asian':    'Asian, NH',
    'Hispanic': 'Hispanic/Latinx',
    'White':    'White, NH',
    'Other':    'Other, NH',
}

REGION_LEVELS = ['Midwest', 'South', 'West', 'Northeast']

AGE_BINS   = [-np.inf, 45, 55, 65, np.inf]
AGE_LABELS = ['- 18-45', '- 45-54', '- 55-64', '- 65+']

A1C_BINS   = [-np.inf, 6.4, 8.0, 9.0, np.inf]
A1C_LABELS = ['- <6.4', '- 6.5-7.9', '- 8.0-8.9', '- >= 9.0']

BMI_BINS   = [-np.inf, 24.9, 29.9, 34.9, 39.9, np.inf]
BMI_LABELS = ['- <24.9', '- 25.0-29.9', '- 30.0-34.9', '- 35.0-39.9', '- >= 40']

WINDOW_END = pd.Timestamp('2019-06-01')

In [ ]:
# =========================
# SMD Functions
# =========================

def smd_binary(n1, d1, n2, d2):
    """SMD for a binary variable (proportions)."""
    p1 = n1 / d1 if d1 > 0 else 0
    p2 = n2 / d2 if d2 > 0 else 0
    pooled = np.sqrt((p1 * (1 - p1) + p2 * (1 - p2)) / 2)
    if pooled == 0:
        return 0.0
    return round(abs(p1 - p2) / pooled, 3)

def smd_continuous(mean1, std1, mean2, std2):
    """SMD for a continuous variable."""
    if pd.isna(mean1) or pd.isna(mean2) or pd.isna(std1) or pd.isna(std2):
        return np.nan
    pooled = np.sqrt((std1**2 + std2**2) / 2)
    if pooled == 0:
        return 0.0
    return round(abs(mean1 - mean2) / pooled, 3)

def fmt(n, denom):
    """Format count as 'n=X, Y.Z%'."""
    if pd.isna(n):
        n = 0
    n = int(n)
    pct = (n / denom) * 100 if denom > 0 else 0.0
    return f'n={n:,}, {pct:.{PCT_DECIMALS}f}%'

In [ ]:
# =========================
# Group Statistics Function
# =========================

def compute_group_stats(dem_g, a1c_g, bmi_g, com_g):
    """
    Compute all Table 1 statistics for a single group.
    Returns a dict of {label: value} for counts, means, and SDs.
    """
    n = len(dem_g)
    stats = {'N': n}

    # --- Female sex ---
    stats['Female sex'] = int(
        dem_g['sex'].astype(str).str.upper().str.strip()
        .isin(['F', 'FEMALE']).sum()
    )

    # --- Age ---
    dem_g = dem_g.copy()
    dem_g['age_2019'] = 2019 - pd.to_numeric(dem_g['year_of_birth'], errors='coerce')
    age_cuts = pd.cut(
        dem_g['age_2019'], bins=AGE_BINS,
        labels=[l.replace('- ', '') for l in AGE_LABELS], right=False
    )
    age_cnt = age_cuts.value_counts()
    for edge_lbl, display_lbl in zip(age_cnt.index.categories, AGE_LABELS):
        stats[display_lbl] = int(age_cnt.get(edge_lbl, 0))
    stats['age_mean'] = dem_g['age_2019'].mean()
    stats['age_std'] = dem_g['age_2019'].std()

    # --- Race/ethnicity ---
    race_series = dem_g['race/ethnicity'].astype(str).str.strip()
    for in_lbl in RACE_INPUT_LEVELS:
        out_lbl = RACE_OUTPUT_MAP[in_lbl]
        stats[f'- {out_lbl}'] = int((race_series == in_lbl).sum())

    # --- US Region ---
    region_series = dem_g['patient_regional_location'].astype(str).str.strip()
    for r in REGION_LEVELS:
        stats[f'- {r}'] = int((region_series == r).sum())

    # --- Baseline HbA1c ---
    a1c_vals = pd.to_numeric(a1c_g['baseline_a1c_value'], errors='coerce')
    stats['a1c_missing'] = int(a1c_vals.isna().sum())
    a1c_valid = a1c_vals.dropna()
    stats['a1c_mean'] = a1c_valid.mean() if len(a1c_valid) > 0 else np.nan
    stats['a1c_std'] = a1c_valid.std() if len(a1c_valid) > 0 else np.nan
    a1c_binned = pd.cut(
        a1c_valid, bins=A1C_BINS,
        labels=[l.replace('- ', '') for l in A1C_LABELS], right=False
    )
    a1c_cnt = a1c_binned.value_counts()
    for edge_lbl, display_lbl in zip(a1c_cnt.index.categories, A1C_LABELS):
        stats[display_lbl] = int(a1c_cnt.get(edge_lbl, 0))

    # --- Baseline BMI ---
    bmi_vals = pd.to_numeric(bmi_g['baseline_bmi_value'], errors='coerce')
    stats['bmi_missing'] = int(bmi_vals.isna().sum())
    bmi_valid = bmi_vals.dropna()
    stats['bmi_mean'] = bmi_valid.mean() if len(bmi_valid) > 0 else np.nan
    stats['bmi_std'] = bmi_valid.std() if len(bmi_valid) > 0 else np.nan
    bmi_binned = pd.cut(
        bmi_valid, bins=BMI_BINS,
        labels=[l.replace('- ', '') for l in BMI_LABELS], right=True
    )
    bmi_cnt = bmi_binned.value_counts()
    for edge_lbl, display_lbl in zip(bmi_cnt.index.categories, BMI_LABELS):
        stats[display_lbl] = int(bmi_cnt.get(edge_lbl, 0))

    # --- Comorbidities ---
    com_g = com_g.copy()
    com_g['date'] = pd.to_datetime(com_g['date'], errors='coerce')
    window = com_g[com_g['date'] <= WINDOW_END].copy()
    for col in ['HF', 'CKD']:
        window[col] = (
            window[col].astype(str).str.strip().str.upper()
            .isin(['TRUE', '1', 'T', 'Y', 'YES'])
        )
    stats['Heart Failure'] = int(
        window.loc[window['HF'], 'patient_id'].astype(str).nunique()
    )
    stats['Chronic Kidney Disease'] = int(
        window.loc[window['CKD'], 'patient_id'].astype(str).nunique()
    )

    return stats

## Section 5 — Compute Statistics for Both Groups

In [ ]:
primary_stats = compute_group_stats(
    dem[dem['group'] == 'Primary'],
    a1c[a1c['group'] == 'Primary'],
    bmi[bmi['group'] == 'Primary'],
    com[com['group'] == 'Primary']
)

disengage_stats = compute_group_stats(
    dem[dem['group'] == 'Disengagement'],
    a1c[a1c['group'] == 'Disengagement'],
    bmi[bmi['group'] == 'Disengagement'],
    com[com['group'] == 'Disengagement']
)

n_p = primary_stats['N']
n_d = disengage_stats['N']

print(f"Primary cohort stats computed: N = {n_p:,}")
print(f"Disengagement cohort stats computed: N = {n_d:,}")

## Section 6 — Assemble Comparison Table

In [ ]:
rows = []

# --- N ---
rows.append(('N', f'{n_p:,}', f'{n_d:,}', ''))

# --- Demographics ---
rows.append(('Demographics', '', '', ''))

rows.append(('- Female sex',
    fmt(primary_stats['Female sex'], n_p),
    fmt(disengage_stats['Female sex'], n_d),
    f"{smd_binary(primary_stats['Female sex'], n_p, disengage_stats['Female sex'], n_d):.3f}"
))

rows.append(('Age (years), mean \u00b1 SD',
    f"{primary_stats['age_mean']:.1f} \u00b1 {primary_stats['age_std']:.1f}",
    f"{disengage_stats['age_mean']:.1f} \u00b1 {disengage_stats['age_std']:.1f}",
    f"{smd_continuous(primary_stats['age_mean'], primary_stats['age_std'], disengage_stats['age_mean'], disengage_stats['age_std']):.3f}"
))

rows.append(('Age category, years', '', '', ''))
for lbl in AGE_LABELS:
    rows.append((lbl,
        fmt(primary_stats[lbl], n_p),
        fmt(disengage_stats[lbl], n_d),
        f"{smd_binary(primary_stats[lbl], n_p, disengage_stats[lbl], n_d):.3f}"
    ))

rows.append(('Race/ethnicity', '', '', ''))
for in_lbl in RACE_INPUT_LEVELS:
    out_lbl = f'- {RACE_OUTPUT_MAP[in_lbl]}'
    rows.append((out_lbl,
        fmt(primary_stats[out_lbl], n_p),
        fmt(disengage_stats[out_lbl], n_d),
        f"{smd_binary(primary_stats[out_lbl], n_p, disengage_stats[out_lbl], n_d):.3f}"
    ))

rows.append(('US Region', '', '', ''))
for r in REGION_LEVELS:
    lbl = f'- {r}'
    rows.append((lbl,
        fmt(primary_stats[lbl], n_p),
        fmt(disengage_stats[lbl], n_d),
        f"{smd_binary(primary_stats[lbl], n_p, disengage_stats[lbl], n_d):.3f}"
    ))

# --- Clinical Characteristics ---
rows.append(('Clinical Characteristics', '', '', ''))

rows.append(('Baseline HbA1c (pre-treatment, 2019)', '', '', ''))
a1c_mean_smd = smd_continuous(
    primary_stats['a1c_mean'], primary_stats['a1c_std'],
    disengage_stats['a1c_mean'], disengage_stats['a1c_std']
)
a1c_mean_p = f"{primary_stats['a1c_mean']:.1f} \u00b1 {primary_stats['a1c_std']:.1f}" if not pd.isna(primary_stats['a1c_mean']) else 'N/A'
a1c_mean_d = f"{disengage_stats['a1c_mean']:.1f} \u00b1 {disengage_stats['a1c_std']:.1f}" if not pd.isna(disengage_stats['a1c_mean']) else 'N/A'
rows.append(('  Mean \u00b1 SD (%)', a1c_mean_p, a1c_mean_d,
    f"{a1c_mean_smd:.3f}" if not pd.isna(a1c_mean_smd) else 'N/A'
))
for lbl in A1C_LABELS:
    rows.append((lbl,
        fmt(primary_stats.get(lbl, 0), n_p),
        fmt(disengage_stats.get(lbl, 0), n_d),
        f"{smd_binary(primary_stats.get(lbl, 0), n_p, disengage_stats.get(lbl, 0), n_d):.3f}"
    ))
rows.append(('- Missing',
    fmt(primary_stats['a1c_missing'], n_p),
    fmt(disengage_stats['a1c_missing'], n_d),
    f"{smd_binary(primary_stats['a1c_missing'], n_p, disengage_stats['a1c_missing'], n_d):.3f}"
))

rows.append(('Baseline BMI (pre-treatment, 2019, kg/m\u00b2)', '', '', ''))
bmi_mean_smd = smd_continuous(
    primary_stats['bmi_mean'], primary_stats['bmi_std'],
    disengage_stats['bmi_mean'], disengage_stats['bmi_std']
)
bmi_mean_p = f"{primary_stats['bmi_mean']:.1f} \u00b1 {primary_stats['bmi_std']:.1f}" if not pd.isna(primary_stats['bmi_mean']) else 'N/A'
bmi_mean_d = f"{disengage_stats['bmi_mean']:.1f} \u00b1 {disengage_stats['bmi_std']:.1f}" if not pd.isna(disengage_stats['bmi_mean']) else 'N/A'
rows.append(('  Mean \u00b1 SD (kg/m\u00b2)', bmi_mean_p, bmi_mean_d,
    f"{bmi_mean_smd:.3f}" if not pd.isna(bmi_mean_smd) else 'N/A'
))
for lbl in BMI_LABELS:
    rows.append((lbl,
        fmt(primary_stats.get(lbl, 0), n_p),
        fmt(disengage_stats.get(lbl, 0), n_d),
        f"{smd_binary(primary_stats.get(lbl, 0), n_p, disengage_stats.get(lbl, 0), n_d):.3f}"
    ))
rows.append(('- Missing',
    fmt(primary_stats['bmi_missing'], n_p),
    fmt(disengage_stats['bmi_missing'], n_d),
    f"{smd_binary(primary_stats['bmi_missing'], n_p, disengage_stats['bmi_missing'], n_d):.3f}"
))

# --- Comorbidities ---
rows.append(('Comorbidities before June 1, 2019', '', '', ''))
rows.append(('- Heart Failure',
    fmt(primary_stats['Heart Failure'], n_p),
    fmt(disengage_stats['Heart Failure'], n_d),
    f"{smd_binary(primary_stats['Heart Failure'], n_p, disengage_stats['Heart Failure'], n_d):.3f}"
))
rows.append(('- Chronic Kidney Disease',
    fmt(primary_stats['Chronic Kidney Disease'], n_p),
    fmt(disengage_stats['Chronic Kidney Disease'], n_d),
    f"{smd_binary(primary_stats['Chronic Kidney Disease'], n_p, disengage_stats['Chronic Kidney Disease'], n_d):.3f}"
))

# --- Create final dataframe ---
comparison_df = pd.DataFrame(rows, columns=[
    'Characteristic',
    f'Primary Cohort (n={n_p:,})',
    f'Disengagement (n={n_d:,})',
    'SMD'
])

comparison_df

## Section 7 — Export

In [ ]:
OUT_CSV  = '/content/eTable_comparison_primary_vs_disengagement.csv'
OUT_XLSX = '/content/eTable_comparison_primary_vs_disengagement.xlsx'

comparison_df.to_csv(OUT_CSV, index=False)

with pd.ExcelWriter(OUT_XLSX, engine='xlsxwriter') as writer:
    sheet = 'Comparison'
    startrow = 2

    comparison_df.to_excel(writer, sheet_name=sheet, index=False, startrow=startrow)

    wb = writer.book
    ws = writer.sheets[sheet]

    title_fmt  = wb.add_format({'bold': True, 'font_size': 14, 'align': 'left'})
    header_fmt = wb.add_format({'bold': True, 'font_size': 12, 'bottom': 1})
    body_fmt   = wb.add_format({'font_size': 11})
    bold_row   = wb.add_format({'font_size': 11, 'bold': True})

    ws.merge_range(0, 0, 0, 3,
        'Comparison of Primary Analytic Cohort vs. Early Disengagement Clusters',
        title_fmt)

    for i, col_name in enumerate(comparison_df.columns):
        ws.write(startrow, i, col_name, header_fmt)

    ws.set_column(0, 0, 45, body_fmt)
    ws.set_column(1, 2, 25, body_fmt)
    ws.set_column(3, 3, 10, body_fmt)

    categories_to_bold = {
        'Demographics',
        'Clinical Characteristics',
        'Comorbidities before June 1, 2019',
    }
    for i, val in enumerate(comparison_df.iloc[:, 0].tolist(), start=1):
        if val in categories_to_bold:
            for c in range(4):
                ws.write(startrow + i, c,
                    comparison_df.iloc[i - 1, c] if c < len(comparison_df.columns) else '',
                    bold_row)

print(f'Exported comparison table to:')
print(f'  CSV:  {OUT_CSV}')
print(f'  XLSX: {OUT_XLSX}')

## Section 8 — Interpretation Notes

**How to interpret the results:**

- **SMD < 0.10:** The two groups are essentially balanced on this characteristic. The post-hoc separation did not introduce meaningful selection bias for this variable.
- **SMD 0.10 – 0.20:** Small imbalance. Worth noting but unlikely to substantially bias downstream analyses.
- **SMD > 0.20:** Meaningful imbalance. This variable differs substantially between the two groups, and the exclusion of disengagement clusters may have introduced selection bias. The direction and implications should be discussed in the Limitations section.

**Key variables to watch:**
- Race/ethnicity: If any racial group is overrepresented in disengagement clusters, the primary cohort's logistic regression ORs may understate real-world prescribing differences.
- HbA1c/BMI missing rates: Higher missingness in the disengagement group is expected (less engagement = fewer labs) but confirms that these patients differ systematically.
- HF/CKD: If comorbidity burden differs, it may affect interpretation of SGLT2i and GLP-1 RA cluster composition.

**For the response letter:**
- If most SMDs < 0.10: State that the two groups were broadly similar in demographic and clinical characteristics, suggesting minimal selection bias.
- If SMDs > 0.20 for specific variables: Acknowledge the imbalance and note the direction of potential bias in the Limitations section.